# Fine-Tuning de Depth-Anything-V2 avec LoRA
**Réalisé par : Marwa Ezzouhri**

Ce notebook applique la technique **LoRA (Low-Rank Adaptation)** pour le fine-tuning du modèle préentraîné **Depth-Anything-V2**, afin de générer des cartes de profondeur précises à partir d'images RGB de pneus.

## Étape 1 : Installation des dépendances

In [ ]:
!pip install torch torchvision transformers peft numpy Pillow matplotlib scikit-learn tqdm -q

## Étape 2 : Imports

In [ ]:
import os
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from transformers import AutoImageProcessor, AutoModelForDepthEstimation
from peft import LoraConfig, get_peft_model

print('Imports OK')

## Étape 3 : Configuration des chemins

In [ ]:
# ── Modifiez ces chemins selon votre machine ──
dataset_path = r'C:\Users\marwa\DATASET_DEVOIR'   # <-- chemin vers votre dataset
output_path  = r'C:\Users\marwa\DepthLoRA_output'  # <-- dossier de sortie (créé automatiquement)

images_dir = os.path.join(dataset_path, 'images')
depth_dir  = os.path.join(dataset_path, 'depth')
os.makedirs(output_path, exist_ok=True)

assert os.path.exists(images_dir), f'Introuvable : {images_dir}'
assert os.path.exists(depth_dir),  f'Introuvable : {depth_dir}'
print('Chemins OK')

## Étape 4 : Chargement et prétraitement des données

In [ ]:
MODEL_ID   = 'depth-anything/Depth-Anything-V2-Small-hf'
IMG_SIZE   = 256
BATCH_SIZE = 4
VAL_SPLIT  = 0.2

processor = AutoImageProcessor.from_pretrained(MODEL_ID)


def extract_depth_from_npy(path, target_size=(IMG_SIZE, IMG_SIZE)):
    """
    Charge un fichier .npy et extrait la carte de profondeur (valeurs Z).
    Gère les cas : (H,W,3), (N,3), (H,W) ou (H,W,1).
    """
    data = np.load(path).astype(np.float32)

    if data.ndim == 3 and data.shape[2] == 3:   # (H, W, 3) -> Z = canal 2
        depth = data[:, :, 2]
    elif data.ndim == 2 and data.shape[1] == 3: # (N, 3)    -> Z = colonne 2
        n = data.shape[0]
        side = int(np.sqrt(n))
        depth = data[:side*side, 2].reshape(side, side)
    elif data.ndim == 3 and data.shape[2] == 1: # (H, W, 1)
        depth = data[:, :, 0]
    else:                                        # (H, W)
        depth = data

    # Supprimer les valeurs aberrantes
    depth = np.nan_to_num(depth, nan=0.0, posinf=0.0, neginf=0.0)
    valid = depth[depth > 0]
    if len(valid) > 0:
        p1, p99 = np.percentile(valid, 1), np.percentile(valid, 99)
        depth = np.clip(depth, p1, p99)

    # Normalisation [0, 1]
    dmin, dmax = depth.min(), depth.max()
    if dmax > dmin:
        depth = (depth - dmin) / (dmax - dmin)

    # Redimensionner
    depth_img = Image.fromarray((depth * 255).astype(np.uint8))
    depth_img = depth_img.resize(target_size, Image.BILINEAR)
    depth = np.array(depth_img).astype(np.float32) / 255.0

    return depth


class DepthDataset(Dataset):
    def __init__(self, images_dir, depth_dir, processor):
        self.images_dir  = images_dir
        self.depth_dir   = depth_dir
        self.processor   = processor
        self.image_files = sorted([f for f in os.listdir(images_dir) if f.endswith('.png')])
        self.depth_files = sorted([f for f in os.listdir(depth_dir)  if f.endswith('.npy')])
        assert len(self.image_files) == len(self.depth_files), \
            'Nombre d images et de fichiers depth different !'
        print(f'{len(self.image_files)} paires image/depth chargees.')

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        # Image RGB
        img = Image.open(os.path.join(self.images_dir, self.image_files[idx])).convert('RGB')
        inputs = self.processor(images=img, return_tensors='pt')
        pixel_values = inputs['pixel_values'].squeeze(0)  # (3, H, W)

        # Carte de profondeur (valeurs Z)
        depth = extract_depth_from_npy(os.path.join(self.depth_dir, self.depth_files[idx]))
        depth_tensor = torch.tensor(depth).unsqueeze(0)   # (1, H, W)

        return pixel_values, depth_tensor


dataset  = DepthDataset(images_dir, depth_dir, processor)
val_size = int(len(dataset) * VAL_SPLIT)
train_size = len(dataset) - val_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f'Train : {train_size} | Val : {val_size}')

## Étape 5 : Chargement de Depth-Anything-V2 + intégration de LoRA

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositif : {device}')

# ── Charger Depth-Anything-V2 ──
base_model = AutoModelForDepthEstimation.from_pretrained(MODEL_ID)

# ── Configuration LoRA ──
# On cible les couches d'attention (query et value) du backbone ViT
lora_config = LoraConfig(
    r=16,                          # rang des matrices de faible rang
    lora_alpha=32,                 # facteur de mise à l'échelle
    target_modules=['query', 'value'],  # couches linéaires ciblées
    lora_dropout=0.1,
    bias='none',
)

model = get_peft_model(base_model, lora_config)
model.to(device)

# Afficher le nombre de paramètres entraînables vs total
model.print_trainable_parameters()

## Étape 6 : Fonction de perte et optimiseur

In [ ]:
def silog_loss(pred, target, variance_focus=0.85):
    """
    Scale-Invariant Log Loss — standard pour l'estimation de profondeur.
    Insensible aux changements d'échelle globaux.
    """
    mask = target > 0
    d = torch.log(pred[mask] + 1e-7) - torch.log(target[mask] + 1e-7)
    return torch.sqrt((d ** 2).mean() - variance_focus * (d.mean() ** 2) + 1e-7)


optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=1e-2
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
print('Optimiseur et scheduler prêts.')

## Étape 7 : Fine-Tuning

In [ ]:
NUM_EPOCHS  = 100
best_val    = float('inf')
train_losses, val_losses = [], []

for epoch in range(NUM_EPOCHS):

    # ── Entraînement ──
    model.train()
    running = 0.0
    for pixel_values, depths in tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [Train]', leave=False):
        pixel_values = pixel_values.to(device)
        depths       = depths.to(device)

        outputs = model(pixel_values=pixel_values)
        pred    = outputs.predicted_depth.unsqueeze(1)  # (B, 1, H, W)
        pred    = F.interpolate(pred, size=depths.shape[-2:], mode='bilinear', align_corners=False)

        loss = silog_loss(pred, depths)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running += loss.item()

    train_loss = running / len(train_loader)
    train_losses.append(train_loss)

    # ── Validation ──
    model.eval()
    val_running = 0.0
    with torch.no_grad():
        for pixel_values, depths in tqdm(val_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [Val]', leave=False):
            pixel_values = pixel_values.to(device)
            depths       = depths.to(device)

            outputs = model(pixel_values=pixel_values)
            pred    = outputs.predicted_depth.unsqueeze(1)
            pred    = F.interpolate(pred, size=depths.shape[-2:], mode='bilinear', align_corners=False)

            val_running += silog_loss(pred, depths).item()

    val_loss = val_running / len(val_loader)
    val_losses.append(val_loss)
    scheduler.step()

    print(f'Epoque [{epoch+1:>3}/{NUM_EPOCHS}]  '
          f'Train Loss: {train_loss:.4f}  Val Loss: {val_loss:.4f}')

    # Sauvegarde du meilleur modèle
    if val_loss < best_val:
        best_val = val_loss
        model.save_pretrained(os.path.join(output_path, 'best_model'))
        print(f'  -> Meilleur modele sauvegarde (val loss: {best_val:.4f})')

print('Entraînement terminé.')

## Étape 8 : Courbes de perte

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Perte Entraînement')
plt.plot(val_losses,   label='Perte Validation')
plt.xlabel('Époque')
plt.ylabel('SiLog Loss')
plt.title('Courbe de Perte — Fine-Tuning LoRA')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(output_path, 'loss_curve.png'))
plt.show()
print(f'Perte finale — Train: {train_losses[-1]:.4f} | Val: {val_losses[-1]:.4f}')

## Étape 9 : Évaluation des performances

In [ ]:
def compute_metrics(pred, target, threshold=0.1):
    """
    Calcule MAE, RMSE, MSE (métriques de régression) +
    Accuracy, Precision, Recall, F1 (classification binaire proche/loin).

    Classification : on binarise pred et target par rapport à leur médiane commune.
    Un pixel est 'loin' (1) si depth > médiane, 'proche' (0) sinon.
    """
    mae  = torch.mean(torch.abs(pred - target)).item()
    rmse = torch.sqrt(torch.mean((pred - target) ** 2)).item()
    mse  = torch.mean((pred - target) ** 2).item()

    # Binarisation par la médiane pour classification proche/loin
    median_val = target.median()
    y_true = (target > median_val).int().cpu().numpy().flatten()
    y_pred = (pred   > median_val).int().cpu().numpy().flatten()

    acc  = accuracy_score(y_true,  y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true,    y_pred, zero_division=0)
    f1   = f1_score(y_true,        y_pred, zero_division=0)

    return {'MSE': mse, 'MAE': mae, 'RMSE': rmse,
            'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1}


model.eval()
all_preds, all_targets = [], []

with torch.no_grad():
    for pixel_values, depths in tqdm(val_loader, desc='Evaluation'):
        pixel_values = pixel_values.to(device)
        depths       = depths.to(device)

        outputs = model(pixel_values=pixel_values)
        pred    = outputs.predicted_depth.unsqueeze(1)
        pred    = F.interpolate(pred, size=depths.shape[-2:], mode='bilinear', align_corners=False)

        all_preds.append(pred.cpu())
        all_targets.append(depths.cpu())

all_preds   = torch.cat(all_preds,   dim=0)
all_targets = torch.cat(all_targets, dim=0)

metrics = compute_metrics(all_preds, all_targets)

print('\n=== Résultats sur le jeu de validation ===')
for k, v in metrics.items():
    print(f'  {k:<12}: {v:.4f}')

## Étape 10 : Visualisation des prédictions

In [ ]:
model.eval()
sample_imgs, sample_depths = next(iter(val_loader))

with torch.no_grad():
    outputs = model(pixel_values=sample_imgs.to(device))
    preds   = outputs.predicted_depth.unsqueeze(1)
    preds   = F.interpolate(preds, size=sample_depths.shape[-2:], mode='bilinear', align_corners=False)
    preds   = preds.cpu()

n = min(4, len(sample_imgs))
fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))

# Normaliser axes en tableau 2D même si n=1
if n == 1:
    axes = axes[np.newaxis, :]

for i in range(n):
    # Image RGB originale
    img_np = sample_imgs[i].permute(1, 2, 0).numpy()
    img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-7)
    axes[i, 0].imshow(img_np)
    axes[i, 0].set_title('Image RGB')
    axes[i, 0].axis('off')

    # Carte de profondeur réelle
    axes[i, 1].imshow(sample_depths[i].squeeze().numpy(), cmap='plasma')
    axes[i, 1].set_title('Profondeur réelle (Z)')
    axes[i, 1].axis('off')

    # Carte de profondeur prédite
    axes[i, 2].imshow(preds[i].squeeze().numpy(), cmap='plasma')
    axes[i, 2].set_title('Profondeur prédite (LoRA)')
    axes[i, 2].axis('off')

plt.suptitle('Prédictions — Depth-Anything-V2 + LoRA', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(output_path, 'predictions.png'))
plt.show()

## Étape 11 : Sauvegarde finale du modèle

In [ ]:
save_path = os.path.join(output_path, 'final_model')
model.save_pretrained(save_path)
processor.save_pretrained(save_path)
print(f'Modèle final sauvegardé dans : {save_path}')